OpenQARP supports a Device object which tries the interface other QPUs. If known, a QPU can be simulated by specifying the characteristics into a Device objects.

If not, experimental setups can assist the experimentation for certain research.

In [ ]:
from qarp.devices import Device, NoiseModel, get_all_to_all_architecture
from qarp.engines import QarpEngine
from qarp.algorithms import Sampler
from qarp.blocks import CompositeBlock, HnBlock, LinearEntanglingBlock, ReadoutBlock
from qarp.plotting import plot_histogram


In [ ]:
n_qubits = 4

# Define a circuit to create a GHZ state: 1/sqrt(2) (|0000> + |1111>)
# Create it with additional Hadamard gates to increase the number of gates though H^2 = I
block = CompositeBlock(
    blocks=[
        HnBlock(n_qubits),
        HnBlock(3, target_qubits=[1, 2, 3]),
        LinearEntanglingBlock(n_qubits, circular=False, use_cz=False),
        ReadoutBlock(n_qubits),
    ]
)
block.build()
block.plot(decompose_boxes=True)

### Noise-free simulation

In [ ]:
my_device = Device(n_qubits)  # Noise-free device
my_engine = QarpEngine(device=my_device)
sampler = Sampler(ket=block, n_shots=1000)
my_engine.build([sampler])
counts_noise_free = my_engine.run()[0]

plot_histogram(counts_noise_free, title="Histogram of measurement outcomes", show_all_solutions=True)


### Noisy simulation

In [ ]:
coupling_map = get_all_to_all_architecture(n_qubits)
noise_model = (
    NoiseModel.amplitude_damping(0.1, gate_set="1q")
    + NoiseModel.bit_flip(0.1, gate_set="1q")
)
my_device = Device(
    n_qubits=n_qubits,
    architecture=coupling_map,
    noise_model=noise_model.inner,
)
my_engine = QarpEngine(device=my_device)
sampler = Sampler(ket=block, n_shots=1000)
my_engine.build([sampler])
counts = my_engine.run()[0]

plot_histogram(counts, title="Histogram of measurement outcomes", show_all_solutions=True)
